# Pré-processamento + Baseline Trivial + Primeiro Modelo — RSNA Abdominal Trauma (G7)

**Marco 2 (Semana 2)** do TP1 — Baseline de Projeto de Pesquisa.

Este notebook cobre:
1. Pipeline de pré-processamento completo e versionado (§4.1 do enunciado).
2. Partição por paciente, definida e **congelada** (não deve mudar depois).
3. Baseline trivial (classificador de classe majoritária) para cada alvo.
4. Primeira família de descritores extraída (features de intensidade/histograma) +
   um classificador treinado — correto e reprodutível, não precisa ser bom ainda.
5. Avaliação com múltiplas métricas (AUC-ROC, AUC-PR, sensibilidade, especificidade,
   além de F1/acurácia balanceada) — acurácia isolada não é aceita pelo enunciado.

> Continuação direta do notebook de EDA (`eda_abdominal_trauma_G7.ipynb`). Mesma
> configuração de `DATA_DIR`, mesma semente (`SEED = 42`).


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from scipy import ndimage as ndi

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    f1_score, confusion_matrix, classification_report,
)

%matplotlib inline


## 1. Configuração (igual ao notebook de EDA)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection")

TRAIN_CSV = DATA_DIR / "train_2024.csv"
SERIES_META_CSV = DATA_DIR / "train_series_meta.csv"
IMAGES_DIR = DATA_DIR / "train_images"

N_PATIENTS_SAMPLE = 180  # ajustado (era 60) para garantir ambas as classes em todo split de teste

train_df = pd.read_csv(TRAIN_CSV)
series_meta_df = pd.read_csv(SERIES_META_CSV)

print("train_2024.csv:", train_df.shape)
print("train_series_meta.csv:", series_meta_df.shape)


## 2. Amostra e partição por paciente (CONGELADA)

Reaproveita (ou recria, se não existir) a amostra estratificada do Marco 1.
A partir daqui, a divisão treino/teste é **fixada em arquivo** — não deve ser
recalculada nas próximas semanas, para evitar qualquer inconsistência entre
experimentos.

Importante: a unidade de amostra é o **paciente** (ver `formulacao_problema.md`),
então a divisão abaixo já garante, por construção, que nenhum paciente aparece
simultaneamente em treino e teste — não há necessidade de `GroupKFold` aqui porque
já não há repetição de paciente entre linhas.

> **Se você está reamostrando com um `N_PATIENTS_SAMPLE` novo** (ex: aumentando de
> 60 para 180 pacientes), apague `sample_patient_ids.csv` e `train_test_split.csv`
> da pasta de trabalho do Kaggle antes de rodar esta célula — caso contrário ela
> reaproveita o arquivo antigo e ignora o novo valor. Se mudar o tamanho da amostra,
> re-rode também o notebook de EDA (Semana 1) com o mesmo `N_PATIENTS_SAMPLE`, para
> manter a documentação do Marco 1 consistente com o restante do projeto.
>
> **Se este notebook (Semana 2) está em uma sessão separada do notebook de EDA**
> (Semana 1 anexado como Input, em vez de rodar na mesma sessão): a célula abaixo
> procura automaticamente `sample_patient_ids.csv`/`train_test_split.csv` tanto na
> pasta de trabalho local quanto dentro de `/kaggle/input/` (onde ficam os notebooks
> anexados como fonte de dados), então não é necessário descobrir o caminho exato
> manualmente.

In [ ]:
SAMPLE_CSV = Path("sample_patient_ids.csv")
SPLIT_CSV = Path("train_test_split.csv")

def find_input_file(filename, search_root=Path("/kaggle/input")):
    """Procura um arquivo pelo nome dentro de /kaggle/input, pulando a pasta de
    dados da competição (que é enorme — centenas de GB de DICOM — e nunca contém
    os CSVs que geramos nós mesmos). Útil quando este notebook é separado do
    notebook que gerou o arquivo, e o Marco 1 foi anexado como Input."""
    if not search_root.exists():
        return None
    for child in search_root.iterdir():
        if child.name == "competitions":
            continue
        matches = list(child.rglob(filename))
        if matches:
            return matches[0]
    return None

strat_col = "any_injury" if "any_injury" in train_df.columns else "bowel_injury"

if SAMPLE_CSV.exists():
    sample_df = pd.read_csv(SAMPLE_CSV)
    print(f"Amostra carregada de {SAMPLE_CSV} ({len(sample_df)} pacientes).")
elif find_input_file("sample_patient_ids.csv") is not None:
    found_path = find_input_file("sample_patient_ids.csv")
    sample_df = pd.read_csv(found_path)
    sample_df.to_csv(SAMPLE_CSV, index=False)  # também salva localmente, p/ próximas células
    print(f"Amostra encontrada em input anexado ({found_path}) e copiada para {SAMPLE_CSV} "
          f"({len(sample_df)} pacientes).")
else:
    sampled_parts = []
    for value, group in train_df.groupby(strat_col):
        n = min(len(group), max(1, round(N_PATIENTS_SAMPLE * len(group) / len(train_df))))
        sampled_parts.append(group.sample(n=n, random_state=SEED))
    sample_df = pd.concat(sampled_parts).reset_index(drop=True)
    sample_df.to_csv(SAMPLE_CSV, index=False)
    print(f"Amostra recriada do zero e salva em {SAMPLE_CSV} ({len(sample_df)} pacientes).")

if SPLIT_CSV.exists():
    split_df = pd.read_csv(SPLIT_CSV)
    print(f"Partição carregada de {SPLIT_CSV}.")
elif find_input_file("train_test_split.csv") is not None:
    found_path = find_input_file("train_test_split.csv")
    split_df = pd.read_csv(found_path)
    split_df.to_csv(SPLIT_CSV, index=False)
    print(f"Partição encontrada em input anexado ({found_path}) e copiada para {SPLIT_CSV}.")
else:
    train_ids, test_ids = train_test_split(
        sample_df["patient_id"],
        test_size=0.2,
        stratify=sample_df[strat_col],
        random_state=SEED,
    )
    split_df = pd.DataFrame({
        "patient_id": sample_df["patient_id"],
        "split": np.where(sample_df["patient_id"].isin(train_ids), "train", "test"),
    })
    split_df.to_csv(SPLIT_CSV, index=False)
    print(f"Partição criada e salva em {SPLIT_CSV} (semente={SEED}).")

print(split_df["split"].value_counts())


## 3. Pipeline de pré-processamento (§4.1)

Etapas, com justificativa:

| Etapa | Por quê |
|---|---|
| Leitura DICOM + conversão para HU | Sem isso, valores de pixel não têm significado físico comparável entre exames (§10.6 do enunciado — erro comum). |
| Janelamento abdominal (C=40, W=400) | Realça a diferenciação entre tecidos moles (fígado, baço, rins, alças intestinais), reduzindo a faixa dinâmica ao intervalo clinicamente relevante. |
| Remoção de mesa/marcadores (crop de corpo) | Estruturas externas ao paciente (mesa de exame, cabos, marcadores) não carregam sinal clínico e podem confundir descritores de textura/forma se não removidas. |
| Redimensionamento para tamanho fixo | Necessário para produzir vetores de features de dimensão consistente entre pacientes com FOV/resolução diferentes. |
| Normalização final [0, 1] | Já embutida no janelamento (contraste linear), mas explicitada aqui como etapa própria do pipeline. |

**Nota sobre reamostragem isotrópica:** dado que a unidade de amostra é o paciente
(2.5D, poucos cortes representativos por paciente — ver `formulacao_problema.md`),
não reamostramos o volume 3D inteiro nesta fase; a resolução no plano (x, y) é
padronizada via redimensionamento. Reamostragem isotrópica completa fica registrada
como limitação a discutir na seção de Metodologia do artigo, caso o grupo decida
avançar para agregação 3D direta numa fase posterior.

In [ ]:
def dicom_to_hu(dcm):
    """Converte pixel_array bruto para Hounsfield Units."""
    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    return dcm.pixel_array.astype(np.float32) * slope + intercept


def apply_window(hu_img, window_center=40, window_width=400):
    """Janelamento clínico, normalizado para [0, 1]."""
    low = window_center - window_width / 2
    high = window_center + window_width / 2
    windowed = np.clip(hu_img, low, high)
    return (windowed - low) / (high - low)


def crop_body(img_01, hu_threshold_frac=0.02):
    """Remove mesa/fundo: mantém só a maior componente conexa acima de um limiar
    de intensidade (o corpo do paciente), e recorta a bounding box dela.
    img_01: imagem já normalizada em [0, 1] (saída de apply_window).
    """
    mask = img_01 > hu_threshold_frac
    if not mask.any():
        return img_01  # fallback: nada a recortar

    labeled, n_components = ndi.label(mask)
    if n_components == 0:
        return img_01

    sizes = ndi.sum(mask, labeled, range(1, n_components + 1))
    largest_label = np.argmax(sizes) + 1
    body_mask = labeled == largest_label

    rows = np.any(body_mask, axis=1)
    cols = np.any(body_mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    cropped = img_01[rmin:rmax + 1, cmin:cmax + 1].copy()
    cropped_mask = body_mask[rmin:rmax + 1, cmin:cmax + 1]
    cropped[~cropped_mask] = 0.0  # zera o que não é corpo (mesa, cabos residuais)
    return cropped


def resize_image(img, output_size=(256, 256)):
    """Redimensiona para tamanho fixo via zoom (interpolação bilinear)."""
    zoom_factors = (output_size[0] / img.shape[0], output_size[1] / img.shape[1])
    return ndi.zoom(img, zoom_factors, order=1)


def preprocess_slice(dcm, window_center=40, window_width=400, output_size=(256, 256)):
    """Pipeline completo: DICOM -> HU -> janela -> crop de corpo -> resize."""
    hu = dicom_to_hu(dcm)
    windowed = apply_window(hu, window_center, window_width)
    cropped = crop_body(windowed)
    resized = resize_image(cropped, output_size)
    return np.clip(resized, 0.0, 1.0)


def load_series_slices(patient_id, series_id, images_dir=IMAGES_DIR):
    series_dir = images_dir / str(patient_id) / str(series_id)
    dcm_paths = sorted(series_dir.glob("*.dcm"), key=lambda p: int(p.stem))
    return [pydicom.dcmread(p) for p in dcm_paths]


In [ ]:
# Demonstração do pipeline completo num paciente de exemplo (versão/teste visual)
example_patient = sample_df["patient_id"].iloc[0]
example_series = series_meta_df.loc[
    series_meta_df["patient_id"] == example_patient, "series_id"
].iloc[0]

slices = load_series_slices(example_patient, example_series)
mid_slice = slices[len(slices) // 2]

processed = preprocess_slice(mid_slice)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(apply_window(dicom_to_hu(mid_slice)), cmap="gray")
axes[0].set_title("Antes (janelado, sem crop/resize)")
axes[1].imshow(processed, cmap="gray")
axes[1].set_title("Depois (pipeline completo)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("preprocessing_before_after.png", dpi=150)
plt.show()

print("Shape final:", processed.shape, "| min/max:", processed.min(), processed.max())


## 4. Agregação corte → paciente (2.5D)

Para cada paciente, selecionamos **N cortes igualmente espaçados** ao longo da série
(cobrindo a extensão abdominal sem processar o volume inteiro) e aplicamos o pipeline
de pré-processamento a cada um. As features extraídas por corte (Seção 5) são então
agregadas por **pooling de média** — a estratégia de agregação definida em
`formulacao_problema.md`.

In [ ]:
N_SLICES_PER_PATIENT = 5

def select_representative_slices(slices, n=N_SLICES_PER_PATIENT):
    """Seleciona n cortes igualmente espaçados ao longo da série."""
    if len(slices) <= n:
        return slices
    idx = np.linspace(0, len(slices) - 1, n).round().astype(int)
    return [slices[i] for i in idx]


## 5. Primeira família de descritores: features de intensidade/histograma

Primeira das ≥3 famílias exigidas pelo §4.2. Simples e rápida de calcular — serve
para validar o pipeline ponta a ponta antes de partir para descritores de textura
mais caros (GLCM, LBP, radiômica) na Semana 3.

Para cada corte pré-processado, calculamos: média, desvio-padrão, assimetria,
curtose e percentis (10, 25, 50, 75, 90) da intensidade — só dentro da região do
corpo (pixels não-zero, já isolados pelo `crop_body`).

In [ ]:
from scipy.stats import skew, kurtosis

def extract_intensity_features(img):
    """Features de intensidade sobre os pixels do corpo (não-zero)."""
    pixels = img[img > 0]
    if pixels.size == 0:
        pixels = img.flatten()

    percentiles = np.percentile(pixels, [10, 25, 50, 75, 90])
    return {
        "intensity_mean": pixels.mean(),
        "intensity_std": pixels.std(),
        "intensity_skew": skew(pixels),
        "intensity_kurtosis": kurtosis(pixels),
        "intensity_p10": percentiles[0],
        "intensity_p25": percentiles[1],
        "intensity_p50": percentiles[2],
        "intensity_p75": percentiles[3],
        "intensity_p90": percentiles[4],
    }


def extract_patient_features(patient_id, series_meta_df, images_dir=IMAGES_DIR):
    """Pipeline completo por paciente: seleciona série, cortes representativos,
    pré-processa cada um, extrai features e agrega por média (pooling)."""
    series_rows = series_meta_df.loc[series_meta_df["patient_id"] == patient_id]
    if series_rows.empty:
        return None
    series_id = series_rows["series_id"].iloc[0]

    slices = load_series_slices(patient_id, series_id, images_dir)
    selected = select_representative_slices(slices)

    per_slice_features = []
    for dcm in selected:
        processed = preprocess_slice(dcm)
        per_slice_features.append(extract_intensity_features(processed))

    feature_df = pd.DataFrame(per_slice_features)
    return feature_df.mean().to_dict()  # pooling por média


In [ ]:
# Extração para toda a amostra (pode levar alguns minutos, dependendo do tamanho)
feature_rows = []
for pid in sample_df["patient_id"]:
    feats = extract_patient_features(pid, series_meta_df)
    if feats is not None:
        feats["patient_id"] = pid
        feature_rows.append(feats)

features_df = pd.DataFrame(feature_rows)
features_df.to_csv("intensity_features.csv", index=False)
print("Features extraídas para", len(features_df), "pacientes.")
features_df.head()


## 5.1 Métricas de avaliação (ampliadas)

O enunciado exige múltiplas medidas além de acurácia (§4.4 — "acurácia isolada não
é aceita"): AUC-ROC, AUC-PR, sensibilidade e especificidade, além de F1 e acurácia
balanceada. A função abaixo centraliza esse cálculo, reaproveitada tanto no baseline
trivial (Seção 6) quanto no primeiro modelo (Seção 7) e nos modelos futuros da
Semana 3 — assim todos os experimentos são avaliados de forma consistente.

In [ ]:
from sklearn.preprocessing import label_binarize

def compute_extended_metrics(y_true, y_pred, y_proba, classes):
    """Calcula acurácia balanceada, F1 macro, AUC-ROC macro, AUC-PR macro,
    sensibilidade média e especificidade média (one-vs-rest) para um alvo.

    y_proba: matriz de probabilidades (saída de predict_proba), colunas na ordem
    de `classes`. Métricas de AUC retornam None quando o teste não tem pelo menos
    2 classes distintas (comum em amostras pequenas para alvos raros).
    """
    result = {
        "balanced_accuracy": round(balanced_accuracy_score(y_true, y_pred), 3),
        "f1_macro": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3),
    }

    if len(classes) >= 2 and len(set(y_true)) >= 2:
        y_true_bin = label_binarize(y_true, classes=classes)
        if y_true_bin.shape[1] == 1:  # binarização de 2 classes vira 1 coluna só
            y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
            y_proba_use = y_proba[:, :2]
        else:
            y_proba_use = y_proba
        try:
            result["auc_roc_macro"] = round(
                roc_auc_score(y_true_bin, y_proba_use, average="macro", multi_class="ovr"), 3
            )
        except ValueError:
            result["auc_roc_macro"] = None
        try:
            result["auc_pr_macro"] = round(
                average_precision_score(y_true_bin, y_proba_use, average="macro"), 3
            )
        except ValueError:
            result["auc_pr_macro"] = None
    else:
        result["auc_roc_macro"] = None
        result["auc_pr_macro"] = None

    cm = confusion_matrix(y_true, y_pred, labels=classes)
    sensitivities, specificities = [], []
    for i in range(len(classes)):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sensitivities.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        specificities.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)

    result["sensitivity_mean"] = round(float(np.nanmean(sensitivities)), 3)
    result["specificity_mean"] = round(float(np.nanmean(specificities)), 3)
    return result


## 6. Baseline trivial

Obrigatório pelo enunciado (§4.3): sem ele não há como interpretar as métricas em
bases desbalanceadas. Classificador de classe majoritária, um por alvo.

In [ ]:
organ_targets = {
    "bowel": ["bowel_healthy", "bowel_injury"],
    "extravasation": ["extravasation_healthy", "extravasation_injury"],
    "kidney": ["kidney_healthy", "kidney_low", "kidney_high"],
    "liver": ["liver_healthy", "liver_low", "liver_high"],
    "spleen": ["spleen_healthy", "spleen_low", "spleen_high"],
}

def get_target_label(row, cols):
    """Converte colunas one-hot (ex: bowel_healthy/bowel_injury) em um rótulo único."""
    present_cols = [c for c in cols if c in row.index]
    active = [c for c in present_cols if row[c] == 1]
    return active[0] if active else present_cols[0]


labeled_df = sample_df.merge(split_df, on="patient_id").merge(features_df, on="patient_id")

for organ, cols in organ_targets.items():
    labeled_df[f"{organ}_label"] = labeled_df.apply(lambda r: get_target_label(r, cols), axis=1)

labeled_df.to_csv("labeled_features.csv", index=False)
print(labeled_df.shape)
labeled_df[[f"{o}_label" for o in organ_targets] + ["split"]].head()


In [ ]:
baseline_results = []

for organ in organ_targets:
    label_col = f"{organ}_label"
    train_mask = labeled_df["split"] == "train"
    test_mask = labeled_df["split"] == "test"

    y_train = labeled_df.loc[train_mask, label_col]
    y_test = labeled_df.loc[test_mask, label_col]

    dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
    dummy.fit(np.zeros((len(y_train), 1)), y_train)  # não usa features de verdade
    y_pred = dummy.predict(np.zeros((len(y_test), 1)))
    y_proba = dummy.predict_proba(np.zeros((len(y_test), 1)))

    metrics = compute_extended_metrics(y_test, y_pred, y_proba, classes=list(dummy.classes_))
    metrics["orgao"] = organ
    metrics["classe_majoritaria"] = y_train.mode().iloc[0]
    baseline_results.append(metrics)

col_order = ["orgao", "classe_majoritaria", "balanced_accuracy", "f1_macro",
             "auc_roc_macro", "auc_pr_macro", "sensitivity_mean", "specificity_mean"]
baseline_df = pd.DataFrame(baseline_results)[col_order]
baseline_df.to_csv("baseline_trivial_results.csv", index=False)
baseline_df


## 7. Primeiro classificador clássico (Regressão Logística)

Primeiro de ≥3 modelos exigidos pelo §4.3. Ainda não é a grade de experimentos
completa (isso fica pra Semana 3) — aqui só confirmamos que o pipeline
features → modelo → métrica funciona de ponta a ponta, de forma correta e
reprodutível.

Todo pré-processamento ajustado a dados (aqui, o `StandardScaler`) é encapsulado em
`Pipeline` e ajustado **somente na partição de treino**, conforme exigido pelo §4.4.

In [ ]:
feature_cols = [c for c in features_df.columns if c != "patient_id"]

model_results = []

for organ in organ_targets:
    label_col = f"{organ}_label"

    X_train = labeled_df.loc[labeled_df["split"] == "train", feature_cols]
    y_train = labeled_df.loc[labeled_df["split"] == "train", label_col]
    X_test = labeled_df.loc[labeled_df["split"] == "test", feature_cols]
    y_test = labeled_df.loc[labeled_df["split"] == "test", label_col]

    # Pula alvos sem exemplos suficientes de alguma classe no split (comum em amostras pequenas)
    if y_train.nunique() < 2 or y_test.nunique() < 1:
        print(f"[{organ}] classes insuficientes no split desta amostra pequena — pulando.")
        continue

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=SEED
        )),
    ])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    metrics = compute_extended_metrics(y_test, y_pred, y_proba, classes=list(clf.classes_))
    metrics["orgao"] = organ
    metrics["modelo"] = "LogisticRegression"
    metrics["n_train"] = len(y_train)
    metrics["n_test"] = len(y_test)
    model_results.append(metrics)

    print(f"--- {organ} ---")
    print(classification_report(y_test, y_pred, zero_division=0))

col_order = ["orgao", "modelo", "balanced_accuracy", "f1_macro", "auc_roc_macro",
             "auc_pr_macro", "sensitivity_mean", "specificity_mean", "n_train", "n_test"]
model_df = pd.DataFrame(model_results)[col_order]
model_df.to_csv("first_model_results.csv", index=False)
model_df


In [ ]:
# Comparação direta: baseline trivial vs. primeiro modelo, lado a lado
comparison_df = baseline_df.merge(
    model_df, on="orgao", suffixes=("_baseline", "_logreg")
)
comparison_df.to_csv("baseline_vs_first_model.csv", index=False)
comparison_df


## 8. Próximos passos

- Extrair as demais famílias de descritores (mínimo 3 no total) — textura (GLCM/LBP),
  forma (momentos de Hu), e/ou radiômica (PyRadiomics).
- Rodar a grade de experimentos descritor × modelo com validação cruzada e ajuste de
  hiperparâmetros (mínimo 3 modelos clássicos: SVM, Random Forest, XGBoost/LightGBM).
- Estudo de ablação: qual etapa do pipeline mais contribui para o desempenho?
- Consolidar tabelas e gerar as figuras para a seção de Resultados do artigo.
